# Lecture 1: Threat Model and Truth Boundary

The motivating system is an autonomous economic agent that may move stablecoins. It faces two broad attack classes:

1. Psychological or game-theoretic attacks that trick the agent into harmful financial actions.
2. Implementation attacks against the cryptographic and tooling stack that protects keys, signatures, proofs, and policy gates.

PACTA focuses on the second class. It does not decide trades, call RPC endpoints, manage custody, or build wallets. It evaluates formal-verification-enhanced tooling and decides whether a constrained component can be used in a funds-protecting path.


## Learning Objectives

- Define a threat model for proof-aware cryptographic tooling.
- Explain why lower-layer arithmetic proofs do not imply wallet safety.
- State the strongest current Ed25519-family claim in theorem-boundary language.
- List common exclusions that remain outside the proof artifact.
- Explain why an autonomous agent needs consequences, not just reports.


## The Core Truth Boundary

The strongest current Ed25519-family claim in this project is approximately:

For selected curve25519-dalek / Solana-Ed25519-family Rust code paths already transpiled into Lean, the verified repositories contain Lean-checked certificates for field arithmetic over `F_p`, `p = 2^255 - 19`, and complete twisted Edwards point-operation laws, under explicit invariants and backend constraints.

That is valuable. And since 2026-07-06 the corpus goes much further: all four ed25519 repositories carry a FOUR-TIER signature apex, button-enforced per fork, up to the full lift - the extracted verifier accepts iff the signature's R decompresses to a valid on-curve point equal to [k](-A)+[s]B. What it is still NOT: a wallet proof, a proof of SHA-512, a proof of the wire parsers (their outcomes are hypotheses), a signing-side proof, or a proof of all Solana transaction behavior. Naming both lists - what is proven and what is not - is the entire discipline of this course.


## Proven or High-Value Evidence

A clean R3-style Ed25519 arithmetic result may cover:

- FieldElement51 arithmetic over `F_p` through denotation.
- Panic and overflow freedom under limb-bound preconditions.
- Complete Edwards point operations under `ExtValid` and `OnCurveExt`.
- Implementation laws through denotation.
- Axiom audit expected to show only standard Lean axioms: `propext`, `Classical.choice`, `Quot.sound`.

The exact theorem names matter. In the current target repos, important certificates include:

- `CurveFieldProofs.fieldImplementation`
- `CurveFieldProofs.edwardsImplementation`


## Explicit Exclusions

Do not let a lower-layer theorem leak into claims about:

- Full EdDSA signature verification.
- Complete Scalar52 arithmetic unless separately proven.
- SHA-512.
- Encoding, decoding, and canonicality unless separately proven.
- Rust compiler correctness.
- Charon/Aeneas translation faithfulness.
- Side-channel resistance.
- SIMD, AVX, hardware, zkVM, accelerator, or syscall paths.
- Wallet policy, transaction construction, RPC, chain, oracle, market, or LLM decision safety.

A professional assurance case is often mostly about preventing evidence from being overextended.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src" / "pacta").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from pacta.config import load_config

config = load_config(repo_root / "examples" / "repos.yaml")
for repo in config.repos:
    print(f"{repo.name:32} kind={repo.kind:13} backend={repo.verified_backend}")
    if repo.backend_warning:
        print(f"  backend warning: {repo.backend_warning}")
    if repo.known_status:
        print(f"  known status: {repo.known_status}")


## Consequences

PACTA turns evaluation into operational consequences:

- If evidence is R0/R1/R2, do not build or consume lower-layer crypto capsules.
- If evidence is R3, a constrained lower-layer component capsule may be built.
- If evidence is below R4, wallet demo construction is refused.
- If a third-party attestation is required but not trusted, the score falls to R0.
- If a transparency receipt is required but invalid or absent, the score falls to R0.

This makes verification a gate, not a decorative badge.


## Checkpoint Questions

1. Why does a proof of field arithmetic not prove transaction construction?
2. What would have to be proven before full EdDSA verification could plausibly reach R4?
3. Why is "the proof failed to replay locally" different from "the theorem is false"?
4. Why should a zkVM accelerator path be excluded unless the repo proves otherwise?


## Exercises

- Pick one configured repository from `examples/repos.yaml`. Write three precise claims that PACTA may make about it and three claims PACTA must refuse.
- Rewrite the sentence "this is verified Ed25519" into a theorem-boundary statement that a security reviewer would accept.
- Create a table mapping each exclusion above to the attack class it leaves open.
